# 3.6 Final-test path, rehearsed on 2023
**Purpose.** Run every step of the 2024 test with 2023 in place of 2024, and require it to reproduce the frozen 2023 values exactly. A failure would then show here and not on the test year. Nothing is chosen in this notebook.

**Method.** The steps of the 2024 run, in order:
1. `check_frozen` and `check_references`: the frozen files match the code, and the 2022 references and calendar shares rebuilt from the 2022 series equal the frozen ones.
2. Labels and rows from the frozen references.
3. Every linear and XGBoost model refitted on the 2022 usable calm origins from the frozen settings (`frozen_models`).
4. The SARIMA forecast paths with the frozen 2022 parameters (`frozen_sarima_paths`).
5. All 11 warning scores of the scored year (`score_year`).
6. `replay_validation`: every 2023 AP and highest-CSI warning threshold must equal its frozen value to 1e-9.
7. The warning measures at the frozen thresholds.

**Inputs and outputs.** The district series 2022 and 2023, the frozen files in `models/` and, for one extra check, the saved 2023 scores. Nothing is written, and 2024 is not read.

## 1. Frozen design and references

In [1]:
import time

import numpy as np
import polars as pl

from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, MODELS_DIR, PANEL_YEAR, PROCESSED_DATA_DIR, VALIDATION_YEAR
from madrid_congestion.evaluation import warning_table
from madrid_congestion.features import origin_rows
from madrid_congestion.frozen import (
    FILES,
    SCORES,
    check_frozen,
    check_references,
    frozen_models,
    frozen_sarima_paths,
    load_frozen,
    replay_validation,
    score_year,
)
from madrid_congestion.labels import add_labels, year_grid
from madrid_congestion.sarima import sarima_grid

SCORED_YEAR = VALIDATION_YEAR  # the 2024 run sets this to TEST_YEAR; everything else stays as it is
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

check_frozen()  # every frozen file present, complete and consistent with the code
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, SCORED_YEAR)}
check_references(series[PANEL_YEAR])  # the frozen 2022 references and calendar shares, rebuilt and compared
refs = pl.read_parquet(MODELS_DIR / FILES["references"])  # from here on, only frozen values are used
calendar = pl.read_parquet(MODELS_DIR / FILES["calendar"])
districts = refs["distrito"]
labels = {year: add_labels(year_grid(df, districts, year), refs) for year, df in series.items()}
rows = {(year, minutes): origin_rows(labels[year], districts, minutes) for year in labels for minutes in HORIZONS}
print("frozen design and 2022 references confirmed |", len(districts), "districts |", {key: df.height for key, df in rows.items()}, "usable calm origins")

frozen design and 2022 references confirmed | 19 districts | {(2022, 30): 475434, (2022, 60): 453544, (2023, 30): 482391, (2023, 60): 461884} usable calm origins


## 2. Models refitted from the frozen settings
Three linear and four XGBoost models per horizon, fitted on 2022 with the frozen hyperparameters. No candidate is tried.

In [2]:
start = time.time()
models = frozen_models({minutes: rows[PANEL_YEAR, minutes] for minutes in HORIZONS}, districts)
print(f"{len(models)} models refitted in {(time.time() - start) / 60:.1f} minutes")

14 models refitted in 2.0 minutes


## 3. SARIMA forecast paths from the frozen parameters
One continuous grid from 1 January 2022 to the end of the scored year. Each of the 38 models is filtered over it with its frozen parameters, so a forecast at *t* uses the observations up to *t* only.

In [3]:
start = time.time()
grid = sarima_grid([labels[PANEL_YEAR], labels[SCORED_YEAR]])
paths = frozen_sarima_paths(grid)
print(f"{len(paths)} forecast paths in {(time.time() - start) / 60:.1f} minutes")

38 forecast paths in 15.4 minutes


## 4. Scores of the scored year and the replay check
`score_year` gives all 11 scores for every usable calm origin. `replay_validation` stops the run unless every AP and warning threshold equals its frozen value.

In [4]:
scored = {minutes: score_year(rows[SCORED_YEAR, minutes], minutes, districts, models, calendar, grid, paths) for minutes in HORIZONS}
replay = replay_validation(scored)  # raises on any difference above 1e-9
print("replay passed:", replay.height, "scores x horizons | largest AP difference", replay["difference"].max(), "| largest threshold difference", replay["threshold_difference"].max())
replay.select("minutes", "score", pl.col("ap_replay", "ap_frozen").round(4), "difference", pl.col("threshold_replay").round(6), "threshold_difference")

replay passed: 22 scores x horizons | largest AP difference 0.0 | largest threshold difference 0.0


minutes,score,ap_replay,ap_frozen,difference,threshold_replay,threshold_difference
i64,str,f64,f64,f64,f64,f64
30,"""proximity""",0.2483,0.2483,0.0,0.908243,0.0
30,"""calendar""",0.3521,0.3521,0.0,0.191667,0.0
30,"""linear_occupancy""",0.4149,0.4149,0.0,0.959307,0.0
30,"""linear_flow""",0.3576,0.3576,0.0,1.028972,0.0
30,"""linear_direct""",0.5172,0.5172,0.0,0.218178,0.0
30,"""xgboost_occupancy""",0.6961,0.6961,0.0,1.003616,0.0
30,"""xgboost_flow""",0.5331,0.5331,0.0,1.022491,0.0
30,"""xgboost_direct""",0.7202,0.7202,0.0,0.283188,0.0
30,"""xgboost_direct_no_flow""",0.716,0.716,0.0,0.299582,0.0


**Extra check.** The rescored values against the scores saved by notebooks 3.0 to 3.3: the same origins and targets, and the largest absolute difference of each score.

In [5]:
KEYS = ["distrito", "slot_start"]
frames = [pl.read_parquet(INTERIM_DATA_DIR / f"scores-{source}.parquet").filter(pl.col("year") == SCORED_YEAR) for source in ["linear", "xgboost", "ablation", "sarima"]]
saved = frames[0]
for df in frames[1:]:
    saved = saved.join(df.drop("target"), on=["year", "minutes", *KEYS], how="inner", validate="1:1")
differences = []
for minutes, df in scored.items():
    saved_here = saved.filter(pl.col("minutes") == minutes)
    joined = df.join(saved_here, on=KEYS, how="full", validate="1:1", suffix="_saved")
    assert joined.height == df.height == saved_here.height  # the same origins
    assert (joined["target"] == joined["target_saved"]).all()  # the same targets
    largest = {}
    for name in SCORES:
        replayed, kept = joined[name].to_numpy(), joined[f"{name}_saved"].to_numpy()
        assert np.isfinite(replayed).all() and np.isfinite(kept).all(), name  # no origin without a score (null or NaN)
        largest[name] = float(np.abs(replayed - kept).max())
    differences.append({"minutes": minutes, **largest})
pl.DataFrame(differences)

minutes,proximity,calendar,linear_occupancy,linear_flow,linear_direct,xgboost_occupancy,xgboost_flow,xgboost_direct,xgboost_direct_no_flow,sarima_occupancy,sarima_flow
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Warning measures at the frozen thresholds
The thresholds from `thresholds-2023.json` are applied unchanged, as on 2024. On 2023, this must reproduce the POD, FAR and CSI of paper Table 6.2.

In [6]:
frozen_thresholds = load_frozen(FILES["thresholds"])
measures = pl.concat([
    warning_table(df["target"], {name: df[name] for name in SCORES}, {name: frozen_thresholds[str(minutes)][name]["warning_threshold"] for name in SCORES})
    .select(pl.lit(minutes).alias("minutes"), pl.lit(df["target"].mean()).alias("base_rate"), pl.all())
    for minutes, df in scored.items()
])
measures.with_columns(pl.col("base_rate", "ap", "pod", "far", "csi").round(4), pl.col("warning_threshold").round(6))

minutes,base_rate,score,ap,warning_threshold,pod,far,csi,hits,misses,false_alarms
i32,f64,str,f64,f64,f64,f64,f64,i64,i64,i64
30,0.0174,"""proximity""",0.2483,0.908243,0.4765,0.7287,0.209,3993,4387,10723
30,0.0174,"""calendar""",0.3521,0.191667,0.4611,0.6378,0.2545,3864,4516,6803
30,0.0174,"""linear_occupancy""",0.4149,0.959307,0.5463,0.6261,0.2853,4578,3802,7665
30,0.0174,"""linear_flow""",0.3576,1.028972,0.4143,0.6356,0.2405,3472,4908,6056
30,0.0174,"""linear_direct""",0.5172,0.218178,0.514,0.543,0.3191,4307,4073,5118
30,0.0174,"""xgboost_occupancy""",0.6961,1.003616,0.6632,0.3805,0.4713,5558,2822,3414
30,0.0174,"""xgboost_flow""",0.5331,1.022491,0.5193,0.4941,0.3446,4352,4028,4250
30,0.0174,"""xgboost_direct""",0.7202,0.283188,0.6896,0.3891,0.4791,5779,2601,3681
30,0.0174,"""xgboost_direct_no_flow""",0.716,0.299582,0.6766,0.3754,0.481,5670,2710,3408


**Result.** The test path reproduces 2023 exactly. The frozen design and the rebuilt 2022 references and calendar shares agree with `models/`. All 22 AP values and all 22 warning thresholds equal the frozen ones with a difference of exactly 0, every rescored value equals the saved score of notebooks 3.0 to 3.3, and the warning measures at the frozen thresholds are those of paper Table 6.2.

**Conclusion.** Notebook 4.0 can use this path unchanged. Its own 2023 replay repeats this check before 2024 is read.